# CTC-only ablation training run -- STANDALONE (no GitHub dependency)

Zero dependency on the `Pich09/tuna-ocr` GitHub repo at runtime: the setup
cell below writes every needed line of library code directly to a local
`lib/` directory and imports from there. This is a frozen, self-contained
snapshot -- future bug fixes to the main repo's `recognizer` package will
NOT automatically reach this notebook. See the markdown cell below for the
full list of what's ported vs intentionally dropped.

Works on **Colab**, **Kaggle**, and a **local machine** -- on **CPU, GPU,
or TPU** (auto-detected):
- Colab: mounts Google Drive at `/content/drive/My Drive/tuna-ocr` and checkpoints there.
- Kaggle: checkpoints to `/kaggle/working/tuna-ocr` (persisted as notebook output).
- Local: checkpoints to `checkpoints/` next to this notebook.

**Before running:** add an `HF_TOKEN` secret (Colab: key icon in the left
sidebar; Kaggle: Add-ons > Secrets; local: `export HF_TOKEN=hf_...`). Also
requires the deduplicated dataset already exist on the Hub
(`Panhapich/tuna-ocr-data`) -- see the data-pull cell below.

**Platform settings to check first:**
- **Kaggle**: internet access is off by default -- Settings (right sidebar) >
  Internet > On (needed to reach the Hugging Face Hub, even with no GitHub
  dependency). Also set Settings > Accelerator to GPU/TPU if you want one.
- **Colab**: Runtime > Change runtime type > pick GPU or TPU if you want one.


## This notebook: CTC-only, fully standalone (80,000 steps)

Two things distinguish this from `train_recognizer_ctc_only.ipynb` (the
GitHub-dependent version):

**1. No AR connection at all, architecturally.** `CTCModel` (in the library
code written below) is `ConformerEncoder` + a plain CTC head -- there is no
AR decoder anywhere in the class, not just a zero-weighted one. Training is
a single parallel forward+backward pass (CTC is not autoregressive), so
batch-size auto-probing IS still ported here (unlike the AR-only standalone
notebook, where free-running training makes the probe itself expensive).

**2. Fully standalone.** Every needed line of `recognizer` code (Conformer
encoder stack, the character tokenizer, checkpoint push/pull, environment/
device detection, the width-bucketed data pipeline) is written to `lib/` by
the setup cell below -- `from recognizer... import ...` never happens.
**Not ported**: the shared SentencePiece subword tokenizer (this notebook
only ever needed the character-level tokenizer -- `recognizer/data/
char_vocab.py`'s docstring documents CTC-on-subword as a measured dead
end), and the from-scratch dataset pull/pack/dedup pipeline (requires the
dataset already exist on the Hub, which it does).

**Tokenizer**: character-level, same `CharTokenizer` design as the
GitHub-dependent version -- built fresh from this run's own training data.

**Identity**: fresh weights, `RUN_NAME = "v2_ctc_only_standalone"`.
Checkpoints push to `Panhapich/tuna-ocr` under its own folder
(`ctc_only_standalone/`) -- separate from the GitHub-dependent CTC-only
run's `ctc_only/` folder, since these are independently-initialized weights
and must never be resumed into each other.


## 1. Setup

In [ ]:
import os

def detect_environment():
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/working"):
        return "kaggle"
    if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ:
        return "colab"
    return "local"

ENV = detect_environment()
print("environment:", ENV)


In [ ]:
# Colab and Kaggle both ship torch preinstalled and matched to their runtime (the CUDA
# driver on a GPU runtime, or the torch_xla/libtpu build on a TPU runtime) -- blindly
# `pip install torch` on top of that can silently replace it with a mismatched build.
# Only pip-install torch if it isn't importable at all (a bare local venv); everything
# else this notebook needs (pyarrow, huggingface_hub, editdistance, pandas, Pillow,
# numpy) installs normally regardless of platform.
import importlib.util

torch_before = None
if importlib.util.find_spec("torch") is not None:
    import torch
    torch_before = torch.__version__

!pip install -q pyarrow huggingface_hub editdistance pandas pillow numpy

if torch_before is None:
    print("torch not found -- installing (no preinstalled build to preserve here)")
    !pip install -q torch
else:
    from importlib.metadata import version as _pkg_version
    torch_after = _pkg_version("torch")
    print(f"using preinstalled torch {torch_after} "
          f"(cuda available: {torch.cuda.is_available()}) -- not reinstalled")


### Write standalone library files (no GitHub clone)

In [ ]:
import sys
from pathlib import Path

LIB_DIR = Path("lib")
LIB_DIR.mkdir(exist_ok=True)

_FILES = {
    "common.py": "\"\"\"Standalone config + character tokenizer -- no GitHub dependency. Ported\nfrom recognizer/config.py + recognizer/tokenizer/char_tokenizer.py +\nrecognizer/data/char_vocab.py's build_char_vocab (trimmed to just the char-\nselection logic, since the full CharVocab class isn't needed here -- both\nCTC-only and AR-only standalone notebooks use ONLY the character-level\nCharTokenizer, never the shared SentencePiece subword tokenizer).\"\"\"\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nDEFAULT_CHECKPOINT_ROOT = Path(\"checkpoints\")\n\n\n@dataclass\nclass ModelConfig:\n    img_height: int = 64\n    chunk_width: int = 128\n    chunk_overlap: int = 16\n    d_model: int = 256\n    num_encoder_layers: int = 8\n    encoder_attn_heads: int = 4\n    encoder_conv_kernel: int = 15\n    encoder_ff_expansion: int = 4\n    encoder_dropout: float = 0.1\n    num_decoder_layers: int = 4\n    decoder_attn_heads: int = 4\n\n\n@dataclass\nclass TrainConfig:\n    batch_size: int = 8\n    lr: float = 5e-4\n    min_lr: float = 1e-6\n    warmup_steps: int = 4000\n    max_steps: int = 80_000\n    val_frac: float = 0.02\n    log_every: int = 100\n    ckpt_every: int = 2_000\n    sample_every: int = 500\n    eval_every: int = 500\n    max_eval_samples: int = 512\n    num_samples: int = 3\n    num_workers: int = 0\n    seed: int = 0\n    max_consecutive_oom: int = 20\n\n\ndef build_char_set(texts, min_count: int = 2) -> list:\n    \"\"\"Deterministic char selection: sorted unique chars seen >= min_count\n    times. Ported from recognizer/data/char_vocab.py's build_char_vocab --\n    same algorithm, so a CharTokenizer built from this over the same data\n    produces the identical id mapping the original library code would.\"\"\"\n    counts = {}\n    for t in texts:\n        for c in t:\n            counts[c] = counts.get(c, 0) + 1\n    return sorted(c for c, n in counts.items() if n >= min_count)\n\n\n_CONTROL_NAMES = [\"PAD\", \"BOS\", \"EOS\", \"EOB\"]\n\n\nclass CharTokenizer:\n    \"\"\"Verbatim port of recognizer/tokenizer/char_tokenizer.py -- see that\n    file's docstring for the full rationale (character-level tokenizer for\n    both CTC and AR heads; CTC-on-subword is a measured dead end, see\n    recognizer/data/char_vocab.py's docstring in the main repo).\"\"\"\n    PAD, BOS, EOS, EOB = range(4)\n    _NUM_CONTROL = 4\n\n    def __init__(self, chars: list):\n        self.chars = list(chars)\n        self.pad_id, self.bos_id, self.eos_id, self.eob_id = self.PAD, self.BOS, self.EOS, self.EOB\n        self.c2i = {c: i + self._NUM_CONTROL for i, c in enumerate(self.chars)}\n        self.i2c = {i + self._NUM_CONTROL: c for i, c in enumerate(self.chars)}\n        self.vocab_size = self._NUM_CONTROL + len(self.chars)\n        self._control_ids = {self.pad_id, self.bos_id, self.eos_id, self.eob_id}\n\n    def encode_plain(self, text: str) -> list:\n        return [self.c2i[c] for c in text if c in self.c2i]\n\n    def decode(self, ids, strip_control: bool = True) -> str:\n        out = []\n        for i in ids:\n            if i in self.i2c:\n                out.append(self.i2c[i])\n            elif not strip_control and i in self._control_ids:\n                out.append(f\"<{_CONTROL_NAMES[i]}>\")\n        return \"\".join(out)\n\n    encode = encode_plain\n\n    @property\n    def size(self) -> int:\n        return self.vocab_size + 1\n\n    @property\n    def blank_id(self) -> int:\n        return self.vocab_size\n\n    def to_json(self) -> str:\n        return json.dumps({\"chars\": self.chars}, ensure_ascii=False)\n\n    @classmethod\n    def from_json(cls, blob: str) -> \"CharTokenizer\":\n        return cls(json.loads(blob)[\"chars\"])\n\n    def save(self, path: Path) -> None:\n        Path(path).parent.mkdir(parents=True, exist_ok=True)\n        Path(path).write_text(self.to_json(), encoding=\"utf-8\")\n\n    @classmethod\n    def load(cls, path: Path) -> \"CharTokenizer\":\n        return cls.from_json(Path(path).read_text(encoding=\"utf-8\"))\n",
    "env_utils.py": "\"\"\"Standalone environment/accelerator detection -- verbatim port of\nrecognizer/env_utils.py (only the intra-package `from .config import\nDEFAULT_CHECKPOINT_ROOT` is replaced by a local constant).\"\"\"\nimport glob\nimport importlib.util\nimport os\nfrom pathlib import Path\n\nDEFAULT_CHECKPOINT_ROOT = Path(\"checkpoints\")\nDRIVE_CHECKPOINT_ROOT = Path(\"/content/drive/My Drive/tuna-ocr/checkpoints\")\nKAGGLE_CHECKPOINT_ROOT = Path(\"/kaggle/working/tuna-ocr/checkpoints\")\n\n_TPU_ENV_VARS = (\n    \"COLAB_TPU_ADDR\", \"XRT_TPU_CONFIG\", \"TPU_NAME\", \"TPU_ACCELERATOR_TYPE\",\n    \"TPU_WORKER_ID\", \"TPU_CHIPS_PER_HOST_BOUNDS\", \"TPU_PROCESS_ADDRESSES\",\n)\n\n\ndef detect_environment() -> str:\n    if \"KAGGLE_KERNEL_RUN_TYPE\" in os.environ or Path(\"/kaggle/working\").is_dir():\n        return \"kaggle\"\n    if \"COLAB_RELEASE_TAG\" in os.environ or \"COLAB_GPU\" in os.environ or \"COLAB_TPU_ADDR\" in os.environ:\n        return \"colab\"\n    return \"local\"\n\n\ndef _tpu_present() -> bool:\n    if os.environ.get(\"PJRT_DEVICE\", \"\").upper() == \"TPU\":\n        return True\n    if any(v in os.environ for v in _TPU_ENV_VARS):\n        return True\n    if glob.glob(\"/dev/accel*\"):\n        return True\n    if importlib.util.find_spec(\"torch_xla\") is None:\n        return False\n    try:\n        import torch_xla.runtime as xr  # noqa: PLC0415\n        return xr.device_type() == \"TPU\"\n    except Exception:\n        return False\n\n\ndef detect_accelerator() -> str:\n    import torch\n    if torch.cuda.is_available():\n        return \"cuda\"\n    return \"tpu\" if _tpu_present() else \"cpu\"\n\n\ndef get_torch_device():\n    accel = detect_accelerator()\n    if accel == \"tpu\":\n        os.environ.setdefault(\"PJRT_DEVICE\", \"TPU\")\n        try:\n            import torch_xla  # noqa: PLC0415\n        except ImportError as exc:\n            raise RuntimeError(\n                \"A TPU was detected but torch_xla isn't installed. On Colab/Kaggle, \"\n                \"select the TPU runtime/accelerator (which ships torch_xla \"\n                \"preinstalled) rather than pip-installing it manually.\"\n            ) from exc\n        if hasattr(torch_xla, \"device\"):\n            return torch_xla.device()\n        import torch_xla.core.xla_model as xm  # noqa: PLC0415\n        return xm.xla_device()\n    import torch\n    return torch.device(\"cuda\" if accel == \"cuda\" else \"cpu\")\n\n\ndef describe_accelerator() -> str:\n    accel = detect_accelerator()\n    if accel == \"cuda\":\n        import torch  # noqa: PLC0415\n        name = torch.cuda.get_device_name(0)\n        total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9\n        return f\"cuda: {name} ({total_gb:.1f} GB)\"\n    if accel == \"tpu\":\n        try:\n            import torch_xla.runtime as xr  # noqa: PLC0415\n            return f\"tpu: {xr.device_type()}, {xr.global_runtime_device_count()} device(s) visible\"\n        except Exception:\n            return \"tpu (torch_xla runtime details unavailable)\"\n    return \"cpu (no GPU/TPU detected -- training will be impractically slow at this scale)\"\n\n\ndef get_checkpoint_root(env: str = None) -> Path:\n    env = env or detect_environment()\n    if env == \"colab\":\n        if not Path(\"/content/drive\").is_dir():\n            from google.colab import drive  # noqa: PLC0415\n            drive.mount(\"/content/drive\")\n        DRIVE_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)\n        return DRIVE_CHECKPOINT_ROOT\n    if env == \"kaggle\":\n        KAGGLE_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)\n        return KAGGLE_CHECKPOINT_ROOT\n    DEFAULT_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)\n    return DEFAULT_CHECKPOINT_ROOT\n\n\ndef get_hf_token(env: str = None) -> str:\n    env = env or detect_environment()\n    if env == \"colab\":\n        from google.colab import userdata  # noqa: PLC0415\n        try:\n            return userdata.get(\"HF_TOKEN\")\n        except Exception as exc:\n            raise RuntimeError(\n                \"HF_TOKEN not found in Colab secrets. Add it via the key icon in the \"\n                \"left sidebar (Secrets) and grant this notebook access.\"\n            ) from exc\n    if env == \"kaggle\":\n        from kaggle_secrets import UserSecretsClient  # noqa: PLC0415\n        try:\n            return UserSecretsClient().get_secret(\"HF_TOKEN\")\n        except Exception as exc:\n            raise RuntimeError(\n                \"HF_TOKEN not found in Kaggle secrets. Add it via Add-ons > Secrets \"\n                \"in the notebook editor.\"\n            ) from exc\n    token = os.environ.get(\"HF_TOKEN\")\n    if not token:\n        raise RuntimeError(\n            \"HF_TOKEN environment variable not set. Export it before running training, \"\n            \"e.g. `export HF_TOKEN=hf_...`.\"\n        )\n    return token\n",
    "hf_io.py": "\"\"\"Standalone Hub I/O -- verbatim port of recognizer/hf_push.py (checkpoint\npush/pull, including the path_prefix folder-sharing convention) and the two\nfunctions this notebook needs from real_data/hf_push.py (prebuilt-dataset\npull only -- push_dataset/the from-scratch pull-pack-dedup pipeline aren't\nported; both standalone notebooks require the dataset already exist on the\nHub, matching how they're actually used in practice).\"\"\"\nimport re\nfrom pathlib import Path\n\n_STEP_RE = re.compile(r\"^step_(\\d+)\\.pt$\")\n\nHF_MODEL_REPO_ID = \"Panhapich/tuna-ocr\"\nHF_DATA_REPO_ID = \"Panhapich/tuna-ocr-data\"\n\n\ndef push_checkpoint(local_ckpt_path: Path, token: str, repo_id: str = HF_MODEL_REPO_ID, private: bool = True,\n                    path_prefix: str = \"\") -> str:\n    \"\"\"`path_prefix`: when set (e.g. \"ar_only\"), uploads under that folder\n    instead of the repo root -- lets independent runs share ONE Hub repo\n    without step-numbered filenames colliding.\"\"\"\n    from huggingface_hub import HfApi\n\n    api = HfApi(token=token)\n    api.create_repo(repo_id=repo_id, exist_ok=True, private=private)\n    filename = Path(local_ckpt_path).name\n    path_in_repo = f\"{path_prefix}/{filename}\" if path_prefix else filename\n    return api.upload_file(\n        path_or_fileobj=str(local_ckpt_path), path_in_repo=path_in_repo, repo_id=repo_id, token=token,\n    )\n\n\ndef pull_latest_checkpoint(dest_dir: Path, token: str = None, repo_id: str = HF_MODEL_REPO_ID, path_prefix: str = \"\"):\n    \"\"\"Downloads the highest-step step_*.pt checkpoint (scoped to\n    path_prefix's folder) into dest_dir, or None if none exist yet.\"\"\"\n    from huggingface_hub import HfApi, hf_hub_download\n    from huggingface_hub.utils import RepositoryNotFoundError\n\n    api = HfApi(token=token)\n    try:\n        files = api.list_repo_files(repo_id)\n    except RepositoryNotFoundError:\n        return None\n\n    prefix = f\"{path_prefix}/\" if path_prefix else \"\"\n    steps = [(int(m.group(1)), f) for f in files\n             if f.startswith(prefix) and (m := _STEP_RE.match(f[len(prefix):]))]\n    if not steps:\n        return None\n    _, latest_file = max(steps)\n\n    downloaded = hf_hub_download(repo_id=repo_id, filename=latest_file, token=token)\n    dest_dir = Path(dest_dir)\n    dest_dir.mkdir(parents=True, exist_ok=True)\n    dest_path = dest_dir / Path(latest_file).name\n    if str(Path(downloaded).resolve()) != str(dest_path.resolve()):\n        import shutil\n        shutil.copy(downloaded, dest_path)\n    return dest_path\n\n\ndef dataset_exists_on_hub(repo_id: str = HF_DATA_REPO_ID, token: str = None) -> bool:\n    from huggingface_hub import HfApi\n    from huggingface_hub.utils import RepositoryNotFoundError\n\n    try:\n        HfApi(token=token).dataset_info(repo_id)\n        return True\n    except RepositoryNotFoundError:\n        return False\n\n\ndef pull_dataset(dest_path: Path, token: str = None, repo_id: str = HF_DATA_REPO_ID) -> Path:\n    from huggingface_hub import hf_hub_download\n    import shutil\n\n    downloaded = hf_hub_download(repo_id=repo_id, filename=\"dedup.arrow\", repo_type=\"dataset\", token=token)\n    dest_path = Path(dest_path)\n    dest_path.parent.mkdir(parents=True, exist_ok=True)\n    if str(Path(downloaded).resolve()) != str(dest_path.resolve()):\n        shutil.copy(downloaded, dest_path)\n    return dest_path\n",
    "data_io.py": "\"\"\"Standalone data loading -- verbatim merge of real_data/chunking.py +\nrecognizer/data/{transforms,manifest}.py.\"\"\"\nimport io\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport torch\nfrom PIL import Image\n\n\n@dataclass\nclass Chunk:\n    image: Image.Image\n    x_offset: int\n    width: int\n\n\ndef chunk_image_overlap(img: Image.Image, chunk_width: int, overlap: int) -> list:\n    \"\"\"Slices `img` left-to-right into chunk_width-wide chunks, each\n    overlapping backward into the previous chunk's tail by `overlap` px.\n    The last chunk keeps its regular stride position (narrower than\n    chunk_width, real pixel content only) rather than being shifted flush\n    to the right edge -- see real_data/chunking.py's docstring in the main\n    repo for why that shift was a bug (inflated real overlap past what the\n    encoder's fixed-frame trim assumes).\"\"\"\n    if not (0 <= overlap < chunk_width):\n        raise ValueError(f\"overlap must satisfy 0 <= overlap < chunk_width, got overlap={overlap}, chunk_width={chunk_width}\")\n    w, h = img.size\n    if w <= chunk_width:\n        return [Chunk(image=img, x_offset=0, width=w)]\n    stride = chunk_width - overlap\n    chunks = []\n    x = 0\n    while x < w:\n        end = min(x + chunk_width, w)\n        chunks.append(Chunk(image=img.crop((x, 0, end, h)), x_offset=x, width=end - x))\n        if end == w:\n            break\n        x += stride\n    return chunks\n\n\nBACKGROUND_VALUE = 0.98  # near-white in [0,1], matches data_gen's off-white line backgrounds\n\n\ndef open_image(image_source):\n    if isinstance(image_source, (bytes, bytearray)):\n        return Image.open(io.BytesIO(image_source))\n    return Image.open(image_source)\n\n\ndef load_and_normalize(image_source, target_height: int = 64) -> torch.Tensor:\n    img = open_image(image_source).convert(\"L\")\n    w, h = img.size\n    new_w = max(1, round(w * target_height / h))\n    img = img.resize((new_w, target_height), Image.BILINEAR)\n    arr = np.asarray(img, dtype=np.float32) / 255.0\n    return torch.from_numpy(arr).unsqueeze(0)\n\n\ndef pad_to_width(chunk: torch.Tensor, target_width: int) -> torch.Tensor:\n    _, h, w = chunk.shape\n    if w >= target_width:\n        return chunk\n    pad = torch.full((1, h, target_width - w), BACKGROUND_VALUE, dtype=chunk.dtype)\n    return torch.cat([chunk, pad], dim=2)\n\n\ndef chunk_line_image(image_source, chunk_width: int, chunk_overlap: int, target_height: int = 64):\n    line_tensor = load_and_normalize(image_source, target_height)\n    line_img = Image.fromarray((line_tensor.squeeze(0).numpy() * 255).astype(np.uint8), mode=\"L\")\n    chunks = chunk_image_overlap(line_img, chunk_width=chunk_width, overlap=chunk_overlap)\n    chunk_tensors, valid_widths = [], []\n    for c in chunks:\n        arr = np.asarray(c.image, dtype=np.float32) / 255.0\n        t = torch.from_numpy(arr).unsqueeze(0)\n        valid_widths.append(c.width)\n        chunk_tensors.append(pad_to_width(t, chunk_width))\n    return chunk_tensors, valid_widths\n\n\n@dataclass\nclass Sample:\n    text: str\n    source: str\n    image_path: str = None\n    image_bytes: bytes = None\n\n    @property\n    def image_source(self):\n        return self.image_bytes if self.image_bytes is not None else self.image_path\n\n\ndef load_dedup_manifest(path) -> list:\n    \"\"\"Only the .arrow path is ported -- both standalone notebooks always\n    consume the prebuilt dedup.arrow already pushed to Panhapich/tuna-ocr-data\n    (see hf_io.py's ensure_dataset), never a from-scratch TSV manifest, so\n    that code path is dead weight here and was dropped rather than\n    reconstructed without being able to verify it against source.\"\"\"\n    import pyarrow as pa\n\n    with pa.memory_map(str(path), \"rb\") as source:\n        table = pa.ipc.open_file(source).read_all()\n    text_col = table.column(\"text\")\n    source_col = table.column(\"source\")\n    image_col = table.column(\"image\")\n    samples = []\n    for t, s, img in zip(text_col, source_col, image_col):\n        samples.append(Sample(image_bytes=img.as_py(), text=t.as_py(), source=s.as_py()))\n    return samples\n",
    "data_pipeline.py": "\"\"\"Standalone width-bucketing + batch-moving helpers -- verbatim port of\nrecognizer/data/dataset.py's compute_widths/BucketBatchSampler/move_batch\n(the parts shared identically by both the CTC-only and AR-only notebooks;\neach notebook defines its own small OCRLineDataset/collate_fn on top of\ndata_io.open_image since their target fields differ).\"\"\"\nimport json\nimport zlib\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport torch\nfrom torch.utils.data import Sampler\n\nfrom data_io import open_image\n\n\ndef image_width(sample, img_height: int) -> int:\n    \"\"\"Scaled width (post fixed-height resize) -- what actually sets a\n    line's chunk count and memory footprint.\"\"\"\n    with open_image(sample.image_source) as img:\n        w, h = img.size\n    return max(1, round(w * img_height / max(1, h)))\n\n\ndef compute_widths(samples: list, img_height: int, num_workers: int = 32, cache_path: Path = None) -> list:\n    def _sample_key(s):\n        if s.image_path is not None:\n            return str(s.image_path)\n        return f\"bytes:{len(s.image_bytes)}:{s.text}\"\n\n    key = None\n    if cache_path is not None and len(samples) > 0:\n        checksum = 0\n        for s in samples:\n            checksum = zlib.crc32(_sample_key(s).encode(\"utf-8\"), checksum)\n        key = f\"v3|{len(samples)}|{checksum}\"\n        try:\n            with open(cache_path, encoding=\"utf-8\") as f:\n                blob = json.load(f)\n            if blob.get(\"key\") == key and len(blob.get(\"widths\", ())) == len(samples):\n                return blob[\"widths\"]\n        except (OSError, ValueError):\n            pass\n\n    with ThreadPoolExecutor(max_workers=num_workers) as pool:\n        widths = list(pool.map(lambda s: image_width(s, img_height), samples))\n\n    if key is not None:\n        try:\n            cache_path = Path(cache_path)\n            cache_path.parent.mkdir(parents=True, exist_ok=True)\n            tmp = cache_path.with_name(cache_path.name + \".tmp\")\n            with open(tmp, \"w\", encoding=\"utf-8\") as f:\n                json.dump({\"key\": key, \"widths\": widths}, f)\n            tmp.replace(cache_path)\n        except OSError:\n            pass\n    return widths\n\n\nclass BucketBatchSampler(Sampler):\n    \"\"\"Sorts indices once by (cheap-to-read) image width so each batch has\n    similar chunk counts, minimizing pad waste, then shuffles BATCH order\n    (not sample order) each epoch.\"\"\"\n\n    def __init__(self, dataset, batch_size: int, shuffle: bool = True, generator=None, widths: list = None):\n        self.dataset = dataset\n        self.batch_size = batch_size\n        self.shuffle = shuffle\n        self.generator = generator\n        indexed = sorted(enumerate(widths), key=lambda p: p[1])\n        sorted_idx = [i for i, _ in indexed]\n        self.batches = [sorted_idx[i:i + batch_size] for i in range(0, len(sorted_idx), batch_size)]\n\n    def __iter__(self):\n        order = list(range(len(self.batches)))\n        if self.shuffle:\n            order = torch.randperm(len(order), generator=self.generator).tolist()\n        for i in order:\n            yield self.batches[i]\n\n    def __len__(self):\n        return len(self.batches)\n\n\nHOST_ONLY_BATCH_KEYS = frozenset({\"chunks_per_line\", \"valid_widths\"})\n\n\ndef move_batch(batch: dict, device) -> dict:\n    return {\n        k: (v.to(device) if torch.is_tensor(v) and k not in HOST_ONLY_BATCH_KEYS else v)\n        for k, v in batch.items()\n    }\n",
    "nn_common.py": "\"\"\"Standalone Conformer encoder stack -- verbatim port of\nrecognizer/modules/{attention,positional,conv_subsampling,conformer_block,\nencoder}.py, merged into one file. No architectural changes from the\noriginal -- this is the same encoder both the production model and these\nablation notebooks use.\"\"\"\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass MultiHeadAttention(nn.Module):\n    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):\n        super().__init__()\n        assert d_model % num_heads == 0\n        self.num_heads = num_heads\n        self.head_dim = d_model // num_heads\n        self.q_proj = nn.Linear(d_model, d_model)\n        self.k_proj = nn.Linear(d_model, d_model)\n        self.v_proj = nn.Linear(d_model, d_model)\n        self.out_proj = nn.Linear(d_model, d_model)\n        self.dropout = dropout\n\n    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:\n        b, t, d = x.shape\n        return x.view(b, t, self.num_heads, self.head_dim).transpose(1, 2)\n\n    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, attn_mask=None) -> torch.Tensor:\n        \"\"\"attn_mask: bool, broadcastable to (B, H, Tq, Tk), True = attend.\"\"\"\n        q = self._split_heads(self.q_proj(query))\n        k = self._split_heads(self.k_proj(key))\n        v = self._split_heads(self.v_proj(value))\n        out = F.scaled_dot_product_attention(\n            q, k, v, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0.0,\n        )\n        out = out.transpose(1, 2).reshape(query.shape[0], query.shape[1], -1)\n        return self.out_proj(out)\n\n\nclass SinusoidalPositionalEncoding(nn.Module):\n    def __init__(self, d_model: int, max_len: int = 4096):\n        super().__init__()\n        pe = torch.zeros(max_len, d_model)\n        pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)\n        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))\n        pe[:, 0::2] = torch.sin(pos * div)\n        pe[:, 1::2] = torch.cos(pos * div)\n        self.register_buffer(\"pe\", pe, persistent=False)\n\n    def forward(self, seq_len: int) -> torch.Tensor:\n        return self.pe[:seq_len]\n\n\nclass Conv2dSubsampling(nn.Module):\n    \"\"\"Input (N, 1, img_height, chunk_width) -> (N, T', d_model). Two stride-2\n    Conv2d(kernel=3, padding=1) + ReLU layers reduce both height and width\n    by ~4x; height is collapsed into the feature dimension.\"\"\"\n\n    def __init__(self, img_height: int, d_model: int, conv_channels: int = 128, in_channels: int = 1):\n        super().__init__()\n        self.conv = nn.Sequential(\n            nn.Conv2d(in_channels, conv_channels, kernel_size=3, stride=2, padding=1),\n            nn.ReLU(),\n            nn.Conv2d(conv_channels, conv_channels, kernel_size=3, stride=2, padding=1),\n            nn.ReLU(),\n        )\n        reduced_h = ((img_height + 1) // 2 + 1) // 2\n        self.out_proj = nn.Linear(conv_channels * reduced_h, d_model)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x = self.conv(x)                       # (N, C, H', W')\n        n, c, h, w = x.shape\n        x = x.permute(0, 3, 1, 2).reshape(n, w, c * h)  # (N, W'=T', d_model)\n        return self.out_proj(x)\n\n\nclass FeedForward(nn.Module):\n    def __init__(self, d_model: int, expansion: int, dropout: float):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.LayerNorm(d_model),\n            nn.Linear(d_model, d_model * expansion),\n            nn.SiLU(),\n            nn.Dropout(dropout),\n            nn.Linear(d_model * expansion, d_model),\n            nn.Dropout(dropout),\n        )\n\n    def forward(self, x):\n        return self.net(x)\n\n\nclass ConvModule(nn.Module):\n    \"\"\"Depthwise Conv1d + GLU, 'same' padding -- no causal restriction needed\n    since a chunk's frames are all visible to each other (this is the\n    Conformer encoder, not the AR decoder).\"\"\"\n\n    def __init__(self, d_model: int, kernel_size: int, dropout: float):\n        super().__init__()\n        self.ln = nn.LayerNorm(d_model)\n        self.pointwise1 = nn.Conv1d(d_model, 2 * d_model, kernel_size=1)\n        self.depthwise = nn.Conv1d(\n            d_model, d_model, kernel_size=kernel_size, padding=kernel_size // 2, groups=d_model,\n        )\n        self.bn = nn.BatchNorm1d(d_model)\n        self.pointwise2 = nn.Conv1d(d_model, d_model, kernel_size=1)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x = self.ln(x).transpose(1, 2)          # (B, D, T)\n        x = self.pointwise1(x)\n        x = F.glu(x, dim=1)\n        x = self.depthwise(x)\n        x = self.bn(x)\n        x = F.silu(x)\n        x = self.pointwise2(x)\n        x = self.dropout(x)\n        return x.transpose(1, 2)                # (B, T, D)\n\n\nclass ConformerBlock(nn.Module):\n    def __init__(self, d_model: int, num_heads: int, conv_kernel: int, ff_expansion: int, dropout: float):\n        super().__init__()\n        self.ff1 = FeedForward(d_model, ff_expansion, dropout)\n        self.ln_attn = nn.LayerNorm(d_model)\n        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)\n        self.conv_module = ConvModule(d_model, conv_kernel, dropout)\n        self.ff2 = FeedForward(d_model, ff_expansion, dropout)\n        self.ln_out = nn.LayerNorm(d_model)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x = x + 0.5 * self.ff1(x)\n        attn_in = self.ln_attn(x)\n        x = x + self.self_attn(attn_in, attn_in, attn_in)\n        x = x + self.conv_module(x)\n        x = x + 0.5 * self.ff2(x)\n        return self.ln_out(x)\n\n\nclass ConformerEncoder(nn.Module):\n    \"\"\"Runs the conv-subsampling + Conformer stack per chunk (all chunks in\n    a batch flattened into one dense pass), then stitches per-chunk outputs\n    back into one per-line sequence, trimming each non-first chunk's\n    overlap-derived leading frames so the stitched sequence has no\n    duplicated content. Chunk index == block index by construction.\"\"\"\n\n    def __init__(self, cfg):\n        super().__init__()\n        self.subsampling = Conv2dSubsampling(cfg.img_height, cfg.d_model)\n        self.pos_enc = SinusoidalPositionalEncoding(cfg.d_model)\n        self.blocks = nn.ModuleList([\n            ConformerBlock(cfg.d_model, cfg.encoder_attn_heads, cfg.encoder_conv_kernel,\n                            cfg.encoder_ff_expansion, cfg.encoder_dropout)\n            for _ in range(cfg.num_encoder_layers)\n        ])\n        # frames trimmed from the start of every non-first chunk, matching\n        # the same ~4x reduction the conv frontend applies to chunk_overlap\n        self.overlap_frames = max(0, math.ceil(cfg.chunk_overlap / 4))\n\n    def forward(self, chunk_batch: torch.Tensor, chunks_per_line: torch.Tensor):\n        \"\"\"chunk_batch: (N, 1, img_height, chunk_width) -- all chunks from\n        all lines in the batch, flattened in line order.\n        chunks_per_line: (B,) how many of those N chunks belong to each line.\n        Returns enc_out (B, T'_max, D), enc_lengths (B,), enc_block_ids (B, T'_max).\n        \"\"\"\n        x = self.subsampling(chunk_batch)          # (N, F, D)\n        x = x + self.pos_enc(x.shape[1]).unsqueeze(0)\n        for block in self.blocks:\n            x = block(x)\n\n        device = x.device\n        d_model = x.shape[-1]\n        lines, block_ids_list = [], []\n        offset = 0\n        for n_chunks in chunks_per_line.tolist():\n            line_chunks = x[offset:offset + n_chunks]     # (n_chunks, F, D)\n            offset += n_chunks\n            trimmed = [line_chunks[0]]\n            for c in range(1, n_chunks):\n                trimmed.append(line_chunks[c, self.overlap_frames:])\n            line_seq = torch.cat(trimmed, dim=0)           # (T_line, D)\n            block_ids = torch.cat([\n                torch.full((t.shape[0],), c, dtype=torch.long, device=device)\n                for c, t in enumerate(trimmed)\n            ])\n            lines.append(line_seq)\n            block_ids_list.append(block_ids)\n\n        enc_lengths = torch.tensor([seq.shape[0] for seq in lines], dtype=torch.long, device=device)\n        t_max = int(enc_lengths.max())\n        b = len(lines)\n        enc_out = torch.zeros(b, t_max, d_model, device=device, dtype=x.dtype)\n        enc_block_ids = torch.zeros(b, t_max, dtype=torch.long, device=device)\n        for i, (seq, bids) in enumerate(zip(lines, block_ids_list)):\n            enc_out[i, :seq.shape[0]] = seq\n            enc_block_ids[i, :bids.shape[0]] = bids\n        return enc_out, enc_lengths, enc_block_ids\n",
    "ctc_model.py": "\"\"\"CTCModel: ConformerEncoder + a plain CTC head only -- no AR decoder\nanywhere in this class. Genuinely no architectural connection to AR.\"\"\"\nimport torch.nn as nn\n\nfrom nn_common import ConformerEncoder\n\n\nclass CTCModel(nn.Module):\n    def __init__(self, cfg, ctc_vocab_size: int):\n        super().__init__()\n        self.encoder = ConformerEncoder(cfg)\n        self.ctc_head = nn.Linear(cfg.d_model, ctc_vocab_size)\n\n    def forward(self, chunk_batch, chunks_per_line):\n        enc_out, enc_lengths, _ = self.encoder(chunk_batch, chunks_per_line)\n        return self.ctc_head(enc_out), enc_lengths\n\n    def encode(self, chunk_batch, chunks_per_line):\n        return self.encoder(chunk_batch, chunks_per_line)\n",
    "ctc_dataset.py": "\"\"\"CTC-only Dataset/collate -- only the character-id CTC target is needed,\nno AR target of any kind (no AR decoder exists in this notebook).\"\"\"\nimport torch\nfrom torch.utils.data import Dataset\n\nfrom data_io import chunk_line_image\n\n\nclass CTCLineDataset(Dataset):\n    def __init__(self, samples: list, tokenizer, cfg):\n        self.samples = samples\n        self.tokenizer = tokenizer\n        self.cfg = cfg\n\n    def __len__(self):\n        return len(self.samples)\n\n    def __getitem__(self, idx):\n        sample = self.samples[idx]\n        chunk_tensors, valid_widths = chunk_line_image(\n            sample.image_source, self.cfg.chunk_width, self.cfg.chunk_overlap, self.cfg.img_height,\n        )\n        ctc_ids = self.tokenizer.encode(sample.text)\n        return {\n            \"chunks\": chunk_tensors,\n            \"valid_widths\": valid_widths,\n            \"ctc_target\": torch.tensor(ctc_ids, dtype=torch.long),\n            \"text\": sample.text,\n        }\n\n\ndef make_collate_fn(pad_id: int):\n    def collate_fn(batch: list) -> dict:\n        chunks_per_line = torch.tensor([len(b[\"chunks\"]) for b in batch], dtype=torch.long)\n        all_chunks = torch.cat([torch.stack(b[\"chunks\"]) for b in batch], dim=0)\n        valid_widths = torch.tensor([w for b in batch for w in b[\"valid_widths\"]], dtype=torch.long)\n\n        ctc_lengths = torch.tensor([len(b[\"ctc_target\"]) for b in batch], dtype=torch.long)\n        max_ctc_len = int(ctc_lengths.max())\n        ctc_targets = torch.full((len(batch), max_ctc_len), pad_id, dtype=torch.long)\n        for i, b in enumerate(batch):\n            ctc_targets[i, :len(b[\"ctc_target\"])] = b[\"ctc_target\"]\n\n        return {\n            \"chunks\": all_chunks,\n            \"chunks_per_line\": chunks_per_line,\n            \"valid_widths\": valid_widths,\n            \"ctc_targets\": ctc_targets,\n            \"ctc_lengths\": ctc_lengths,\n            \"texts\": [b[\"text\"] for b in batch],\n        }\n\n    return collate_fn\n",
    "ctc_train.py": "\"\"\"CTC-only training loop -- no AR decoder anywhere. Standalone: no import\nof `recognizer`/`real_data`. Unlike the AR-only notebook, CTC training is a\nsingle parallel forward+backward pass (not autoregressive), so batch-size\nauto-probing is cheap and IS ported here.\"\"\"\nimport csv\nimport gc\nimport logging\nimport math\nimport os\nimport random\nimport time\nfrom pathlib import Path\n\nos.environ.setdefault(\"PYTORCH_ALLOC_CONF\", \"expandable_segments:True\")\nos.environ.setdefault(\"PYTORCH_CUDA_ALLOC_CONF\", \"expandable_segments:True\")\n\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\n\nimport env_utils\nimport hf_io\nfrom ctc_dataset import CTCLineDataset, make_collate_fn\nfrom ctc_model import CTCModel\nfrom common import CharTokenizer, build_char_set\nfrom data_io import chunk_line_image, load_dedup_manifest\nfrom data_pipeline import BucketBatchSampler, compute_widths, move_batch\n\n\ndef build_logger(ckpt_dir: Path, run_name: str) -> logging.Logger:\n    logger = logging.getLogger(f\"ctc_only.{run_name}\")\n    logger.setLevel(logging.INFO)\n    logger.handlers.clear()\n    logger.propagate = False\n    fmt = logging.Formatter(\"%(asctime)s %(message)s\", datefmt=\"%Y-%m-%d %H:%M:%S\")\n    fh = logging.FileHandler(ckpt_dir / \"train.log\")\n    fh.setFormatter(fmt)\n    sh = logging.StreamHandler()\n    sh.setFormatter(fmt)\n    logger.addHandler(fh)\n    logger.addHandler(sh)\n    return logger\n\n\ndef lr_lambda(step: int, warmup_steps: int, max_steps: int, min_lr_ratio: float = 0.0) -> float:\n    if step < warmup_steps:\n        return min_lr_ratio + (1 - min_lr_ratio) * (step / max(1, warmup_steps))\n    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)\n    cosine = 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))\n    return min_lr_ratio + (1 - min_lr_ratio) * cosine\n\n\ndef _is_oom_error(exc) -> bool:\n    if isinstance(exc, torch.cuda.OutOfMemoryError):\n        return True\n    msg = str(exc).lower()\n    return any(s in msg for s in (\n        \"out of memory\", \"device not ready\", \"cuda driver error\",\n        \"cublas_status_alloc_failed\", \"cudnn_status_alloc_failed\",\n    ))\n\n\ndef _collapse_ctc(row: list, blank_id: int) -> list:\n    out, prev = [], None\n    for c in row:\n        if c != prev and c != blank_id:\n            out.append(c)\n        prev = c\n    return out\n\n\ndef find_max_batch_size(model, dataset, collate_fn, device, widths, start: int = 64, min_batch: int = 1) -> int:\n    if device.type != \"cuda\":\n        return start\n    widest_idx = sorted(range(len(widths)), key=lambda i: widths[i], reverse=True)\n    optim_state_bytes = sum(p.numel() for p in model.parameters()) * 2 * 4\n    bs = start\n    ctc_logits = fake_objective = None\n    while bs >= min_batch:\n        try:\n            idxs = widest_idx[:bs]\n            batch = collate_fn([dataset[i] for i in idxs])\n            batch = move_batch(batch, device)\n            ctc_logits, _ = model(batch[\"chunks\"], batch[\"chunks_per_line\"])\n            fake_objective = ctc_logits.float().sum()\n            fake_objective.backward()\n            optim_reserve = torch.empty(optim_state_bytes, dtype=torch.uint8, device=device)\n            del optim_reserve\n            model.zero_grad(set_to_none=True)\n            del ctc_logits, fake_objective\n            torch.cuda.empty_cache()\n            return bs\n        except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:\n            if not _is_oom_error(exc):\n                raise\n            model.zero_grad(set_to_none=True)\n            del ctc_logits, fake_objective\n            gc.collect()\n            torch.cuda.empty_cache()\n            ctc_logits = fake_objective = None\n            bs //= 2\n    return max(min_batch, 1)\n\n\n@torch.no_grad()\ndef evaluate_val_cer(model, tokenizer, model_cfg, val_samples, device, batch_size: int = 16):\n    import editdistance\n\n    def cer(refs, hyps):\n        if not refs:\n            return float(\"nan\")\n        edits = sum(editdistance.eval(r, h) for r, h in zip(refs, hyps))\n        return edits / (sum(len(r) for r in refs) or 1)\n\n    def scaled_width(sample):\n        from data_io import open_image\n        with open_image(sample.image_source) as img:\n            w, h = img.size\n        return max(1, round(w * model_cfg.img_height / max(1, h)))\n\n    was_training = model.training\n    model.eval()\n    ordered = sorted(val_samples, key=scaled_width)\n    refs, hyps = [], []\n    for i in range(0, len(ordered), batch_size):\n        group = ordered[i:i + batch_size]\n        all_chunks, cpl = [], []\n        for s in group:\n            ct, _ = chunk_line_image(s.image_source, model_cfg.chunk_width, model_cfg.chunk_overlap, model_cfg.img_height)\n            all_chunks.extend(ct)\n            cpl.append(len(ct))\n        chunk_batch = torch.stack(all_chunks).to(device)\n        enc_out, enc_lengths, _ = model.encode(chunk_batch, torch.tensor(cpl, dtype=torch.long))\n        ids = model.ctc_head(enc_out).argmax(-1).cpu()\n        lengths = enc_lengths.cpu().tolist()\n        for sample, row, n in zip(group, ids.tolist(), lengths):\n            refs.append(sample.text)\n            hyps.append(tokenizer.decode(_collapse_ctc(row[:n], tokenizer.blank_id)))\n    if was_training:\n        model.train()\n    return cer(refs, hyps), len(refs)\n\n\n@torch.no_grad()\ndef log_inference_samples(model, tokenizer, model_cfg, samples, device, logger, step):\n    was_training = model.training\n    model.eval()\n    for i, sample in enumerate(samples):\n        chunk_tensors, _ = chunk_line_image(sample.image_source, model_cfg.chunk_width, model_cfg.chunk_overlap, model_cfg.img_height)\n        chunk_batch = torch.stack(chunk_tensors).to(device)\n        chunks_per_line = torch.tensor([len(chunk_tensors)], dtype=torch.long)\n        enc_out, enc_lengths, _ = model.encode(chunk_batch, chunks_per_line)\n        n = int(enc_lengths[0])\n        row = model.ctc_head(enc_out).argmax(-1)[0][:n].cpu().tolist()\n        pred = tokenizer.decode(_collapse_ctc(row, tokenizer.blank_id))\n        logger.info(f\"  [sample {i}] step {step} gt={sample.text!r} ctc={pred!r}\")\n    if was_training:\n        model.train()\n\n\ndef run_training(model_cfg, train_cfg, dedup_manifest_path, checkpoint_root, run_name: str,\n                 push_to_hub: bool, repo_id: str, hf_token: str, hub_private: bool,\n                 resume_path, hub_path_prefix: str, auto_batch_size: bool = True, device=None):\n    device = device or env_utils.get_torch_device()\n    torch.manual_seed(train_cfg.seed)\n\n    print(\"run_training: loading samples...\", flush=True)\n    samples = load_dedup_manifest(dedup_manifest_path)\n    if not samples:\n        raise RuntimeError(f\"No samples found at {dedup_manifest_path}\")\n    samples = list(samples)\n    random.Random(train_cfg.seed).shuffle(samples)\n    n_val = max(1, int(len(samples) * train_cfg.val_frac))\n    train_samples, val_samples = samples[n_val:], samples[:n_val]\n\n    ckpt_dir = Path(checkpoint_root) / run_name\n    ckpt_dir.mkdir(parents=True, exist_ok=True)\n    tok_path = ckpt_dir / \"char_tokenizer.json\"\n    print(f\"run_training: {'loading' if tok_path.exists() else 'building'} char tokenizer...\", flush=True)\n    if tok_path.exists():\n        tokenizer = CharTokenizer.load(tok_path)\n    else:\n        chars = build_char_set((s.text for s in train_samples), min_count=2)\n        tokenizer = CharTokenizer(chars)\n        tokenizer.save(tok_path)\n\n    train_ds = CTCLineDataset(train_samples, tokenizer, model_cfg)\n    collate_fn = make_collate_fn(tokenizer.pad_id)\n    inference_samples = val_samples[:train_cfg.num_samples]\n    eval_samples = val_samples[:train_cfg.max_eval_samples] if train_cfg.max_eval_samples else val_samples\n\n    print(f\"run_training: building model + moving to {device}...\", flush=True)\n    model = CTCModel(model_cfg, ctc_vocab_size=tokenizer.size).to(device)\n\n    logger = build_logger(ckpt_dir, run_name)\n    logger.info(f\"run '{run_name}': device={device}, {len(train_samples)} train / {len(val_samples)} val samples, \"\n                f\"checkpoints -> {ckpt_dir}, log file -> {ckpt_dir / 'train.log'}\")\n\n    widths_cache = ckpt_dir / \"widths_cache.json\"\n    t_w = time.time()\n    widths = compute_widths(train_samples, model_cfg.img_height, cache_path=widths_cache)\n    logger.info(f\"image widths ready in {time.time() - t_w:.1f}s (cache: {widths_cache})\")\n\n    batch_size = train_cfg.batch_size\n    if auto_batch_size and device.type == \"cuda\":\n        batch_size = find_max_batch_size(model, train_ds, collate_fn, device, widths, start=max(64, train_cfg.batch_size))\n        logger.info(f\"auto batch size: {batch_size}\")\n    elif auto_batch_size:\n        logger.info(f\"auto batch size requested but device is {device.type} -- using configured batch_size={batch_size}\")\n\n    if train_cfg.warmup_steps >= train_cfg.max_steps:\n        clamped = max(1, int(0.1 * train_cfg.max_steps))\n        logger.info(f\"WARNING: warmup_steps={train_cfg.warmup_steps} >= max_steps={train_cfg.max_steps} -- \"\n                    f\"clamping warmup_steps to {clamped}.\")\n        train_cfg.warmup_steps = clamped\n\n    sampler = BucketBatchSampler(train_ds, batch_size=batch_size, shuffle=True, widths=widths)\n    if train_cfg.num_workers > 0:\n        import multiprocessing as mp\n        try:\n            mp.set_start_method(\"fork\", force=True)\n        except RuntimeError:\n            pass\n    loader = DataLoader(train_ds, batch_sampler=sampler, collate_fn=collate_fn,\n                        num_workers=train_cfg.num_workers, pin_memory=(device.type == \"cuda\"),\n                        persistent_workers=train_cfg.num_workers > 0)\n\n    optimizer = torch.optim.AdamW(model.parameters(), lr=train_cfg.lr)\n    min_lr_ratio = train_cfg.min_lr / train_cfg.lr\n    scheduler = torch.optim.lr_scheduler.LambdaLR(\n        optimizer, lambda step: lr_lambda(step, train_cfg.warmup_steps, train_cfg.max_steps, min_lr_ratio))\n\n    log_path = ckpt_dir / \"train_log.csv\"\n    if not log_path.exists():\n        log_path.write_text(\"step,epoch,loss,lr\\n\")\n    eval_log_path = ckpt_dir / \"eval_log.csv\"\n    if not eval_log_path.exists():\n        eval_log_path.write_text(\"step,epoch,val_ctc_cer\\n\")\n\n    step = 0\n    if resume_path:\n        state = torch.load(resume_path, map_location=\"cpu\", weights_only=False)\n        model.load_state_dict(state[\"model_state_dict\"])\n        optimizer.load_state_dict(state[\"optimizer_state_dict\"])\n        scheduler.load_state_dict(state[\"scheduler_state_dict\"])\n        step = state[\"step\"]\n        logger.info(f\"resumed from {resume_path} at step {step} \"\n                    f\"({train_cfg.max_steps - step} steps remaining to max_steps={train_cfg.max_steps})\")\n\n    steps_per_epoch = max(1, len(train_ds) // batch_size)\n    logger.info(f\"batch_size={batch_size}, ~{steps_per_epoch} steps/epoch, max_steps={train_cfg.max_steps}\")\n\n    oom_skips = 0\n    consecutive_oom = 0\n    ctc_logits = loss = enc_lengths = None\n    model.train()\n    while step < train_cfg.max_steps:\n        for batch in loader:\n            if step >= train_cfg.max_steps:\n                break\n            try:\n                batch = move_batch(batch, device)\n                ctc_logits, enc_lengths = model(batch[\"chunks\"], batch[\"chunks_per_line\"])\n                b, t_max, _ = ctc_logits.shape\n                log_probs = F.log_softmax(ctc_logits, dim=-1).transpose(0, 1)\n                ctc_input_lengths = enc_lengths.to(dtype=torch.long, device=ctc_logits.device)\n                loss = F.ctc_loss(log_probs, batch[\"ctc_targets\"], ctc_input_lengths, batch[\"ctc_lengths\"],\n                                  blank=tokenizer.blank_id, zero_infinity=True)\n                optimizer.zero_grad(set_to_none=True)\n                loss.backward()\n                optimizer.step()\n            except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:\n                if not _is_oom_error(exc):\n                    raise\n                oom_skips += 1\n                consecutive_oom += 1\n                optimizer.zero_grad(set_to_none=True)\n                del ctc_logits, loss, enc_lengths\n                gc.collect()\n                torch.cuda.empty_cache()\n                first_line = str(exc).splitlines()[0] if str(exc) else type(exc).__name__\n                logger.warning(f\"step {step}: CUDA OOM #{consecutive_oom}/{train_cfg.max_consecutive_oom} \"\n                               f\"({oom_skips} total this run) -- skipping. {first_line}\")\n                if consecutive_oom >= train_cfg.max_consecutive_oom:\n                    raise RuntimeError(\n                        f\"{consecutive_oom} consecutive CUDA OOMs at step {step} -- stopping instead of \"\n                        f\"retrying forever. The last checkpoint (step {step - step % train_cfg.ckpt_every}) \"\n                        f\"already has everything before this streak -- resume from it, and reduce \"\n                        f\"TrainConfig.batch_size if this recurs. Last error:\\n{exc}\"\n                    ) from exc\n                ctc_logits = loss = enc_lengths = None\n                continue\n            consecutive_oom = 0\n            scheduler.step()\n            step += 1\n\n            if step % train_cfg.log_every == 0:\n                lr = scheduler.get_last_lr()[0]\n                epoch = step / steps_per_epoch\n                oom_note = f\" oom_skips {oom_skips}\" if oom_skips else \"\"\n                logger.info(f\"step {step} epoch {epoch:.2f} loss {loss.item():.4f} lr {lr:.2e}{oom_note}\")\n                with log_path.open(\"a\", newline=\"\") as f:\n                    csv.writer(f).writerow([step, f\"{epoch:.4f}\", loss.item(), lr])\n\n            if train_cfg.sample_every > 0 and step % train_cfg.sample_every == 0 and inference_samples:\n                log_inference_samples(model, tokenizer, model_cfg, inference_samples, device, logger, step)\n\n            if train_cfg.eval_every > 0 and step % train_cfg.eval_every == 0 and eval_samples:\n                t_eval = time.time()\n                ctc_cer, n_ctc = evaluate_val_cer(model, tokenizer, model_cfg, eval_samples, device)\n                epoch = step / steps_per_epoch\n                logger.info(f\"eval step {step} epoch {epoch:.2f} val_ctc_cer {ctc_cer:.4f} (n={n_ctc}) \"\n                            f\"[{time.time() - t_eval:.1f}s]\")\n                with eval_log_path.open(\"a\", newline=\"\") as f:\n                    csv.writer(f).writerow([step, f\"{epoch:.4f}\", f\"{ctc_cer:.4f}\"])\n\n            if step % train_cfg.ckpt_every == 0:\n                ckpt_path = ckpt_dir / f\"step_{step:07d}.pt\"\n                state = {\n                    \"step\": step, \"model_state_dict\": model.state_dict(),\n                    \"optimizer_state_dict\": optimizer.state_dict(),\n                    \"scheduler_state_dict\": scheduler.state_dict(), \"model_cfg\": model_cfg,\n                    \"train_cfg_warmup_steps\": train_cfg.warmup_steps,\n                    \"train_cfg_max_steps\": train_cfg.max_steps,\n                }\n                torch.save(state, ckpt_path)\n                torch.save(state, ckpt_dir / \"last.pt\")\n                logger.info(f\"saved checkpoint {ckpt_path}\")\n                if push_to_hub:\n                    url = hf_io.push_checkpoint(ckpt_path, token=hf_token, repo_id=repo_id,\n                                                private=hub_private, path_prefix=hub_path_prefix)\n                    logger.info(f\"pushed checkpoint to {url}\")\n\n    logger.info(f\"training complete: {step} steps, checkpoints + log in {ckpt_dir}\"\n                f\"{f', {oom_skips} batches skipped due to CUDA OOM' if oom_skips else ''}\")\n    return model\n",
}

for _name, _src in _FILES.items():
    (LIB_DIR / _name).write_text(_src, encoding="utf-8")

if str(LIB_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(LIB_DIR.resolve()))

print(f"wrote {len(_FILES)} standalone library files to {LIB_DIR.resolve()} -- no GitHub clone needed")


In [ ]:
import os

import env_utils

checkpoint_root = env_utils.get_checkpoint_root(ENV)

# Token resolution, in order: platform secret store, HF_TOKEN env var, a plain file,
# then an interactive prompt -- deliberately never hardcoded in this notebook.
hf_token = None
try:
    hf_token = env_utils.get_hf_token(ENV)
except Exception as e:
    print(f"platform secret store unavailable ({type(e).__name__}), trying fallbacks...")
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    for candidate in ("/content/hf_token.txt", "/kaggle/working/hf_token.txt", "hf_token.txt"):
        if os.path.exists(candidate):
            hf_token = open(candidate).read().strip()
            print(f"read HF token from {candidate}")
            break
if not hf_token:
    import getpass
    hf_token = getpass.getpass("HF token (input hidden): ").strip()

accelerator = env_utils.detect_accelerator()
device = env_utils.get_torch_device()

print("environment:      ", ENV)
print("checkpoint root:  ", checkpoint_root)
print("HF token loaded:  ", bool(hf_token))
print("accelerator:      ", env_utils.describe_accelerator())
print("torch device:     ", device)
if accelerator == "cpu":
    print("\n>>> No GPU/TPU detected. Colab: Runtime > Change runtime type. "
          "Kaggle: Settings > Accelerator. Do not start the training cell on CPU -- "
          "at this dataset's scale it will not finish.")


## 2. Data\n\nDownloads the prebuilt deduplicated dataset from the Hub.

In [ ]:
from pathlib import Path

import hf_io

# Both standalone notebooks require the deduplicated dataset already exist on the Hub
# (Panhapich/tuna-ocr-data) -- it was built once by the original data-pull notebook
# (train_recognizer_v2_scratch.ipynb's section 2), which is NOT ported here (that
# pull -> pack -> dedup pipeline is a separate, large piece of the real_data package
# and isn't needed once the prebuilt dataset already exists, which it does).
dedup_manifest = Path("data") / "dedup.arrow"
dedup_manifest.parent.mkdir(parents=True, exist_ok=True)

if dedup_manifest.exists():
    print(f"{dedup_manifest} already present locally, skipping download")
elif hf_io.dataset_exists_on_hub(hf_io.HF_DATA_REPO_ID, token=hf_token):
    print(f"downloading prebuilt dataset from the Hub ({hf_io.HF_DATA_REPO_ID})...")
    hf_io.pull_dataset(dedup_manifest, token=hf_token, repo_id=hf_io.HF_DATA_REPO_ID)
    print("downloaded:", dedup_manifest)
else:
    raise RuntimeError(
        f"No prebuilt dataset found on {hf_io.HF_DATA_REPO_ID}. This standalone notebook "
        f"doesn't include the from-scratch pull/pack/dedup pipeline (a separate, large "
        f"part of the real_data package) -- run train_recognizer_v2_scratch.ipynb's "
        f"section 2 once first (from the main GitHub repo) to build and push it, then "
        f"re-run this cell."
    )


## 3. Train

In [ ]:
from pathlib import Path

from common import ModelConfig, TrainConfig
import ctc_train
import hf_io

model_cfg = ModelConfig(d_model=256, num_encoder_layers=8)

LOG_EVERY = 100
CKPT_EVERY = 2_000

train_cfg = TrainConfig(
    log_every=LOG_EVERY,
    ckpt_every=CKPT_EVERY,
    max_eval_samples=512,
    max_steps=80_000,
)

RUN_NAME = "v2_ctc_only_standalone"
CHECKPOINT_REPO_ID = "Panhapich/tuna-ocr"
HUB_PATH_PREFIX = "ctc_only_standalone"

resume_path = hf_io.pull_latest_checkpoint(Path(checkpoint_root) / RUN_NAME, token=hf_token,
                                           repo_id=CHECKPOINT_REPO_ID, path_prefix=HUB_PATH_PREFIX)
if resume_path:
    print(f"resuming from the latest checkpoint on the Hub: {resume_path}")
else:
    print(f"no Hub checkpoint found for {CHECKPOINT_REPO_ID}/{HUB_PATH_PREFIX} -- starting {RUN_NAME} from scratch")

model = ctc_train.run_training(
    model_cfg, train_cfg,
    dedup_manifest_path=dedup_manifest,
    checkpoint_root=checkpoint_root,
    run_name=RUN_NAME,
    push_to_hub=True,
    repo_id=CHECKPOINT_REPO_ID,
    hf_token=hf_token,
    hub_private=True,
    resume_path=resume_path,
    hub_path_prefix=HUB_PATH_PREFIX,
    auto_batch_size=True,
)


## 4. Check progress\n\nReads this run's own `eval_log.csv`. Safe to run anytime.

In [ ]:
import pandas as pd

log_path = Path(checkpoint_root) / RUN_NAME / "eval_log.csv"
if not log_path.exists():
    print(f"no eval_log.csv yet at {log_path} -- has training started?")
else:
    df = pd.read_csv(log_path)
    latest = df.iloc[-1]
    metric_col = [c for c in df.columns if c.startswith("val_")][0]
    best = df.loc[df[metric_col].idxmin()]
    print(f"{RUN_NAME}: {len(df)} eval points, latest step {int(latest.step)} (epoch {latest.epoch:.2f})")
    print(f"  latest: {metric_col} {latest[metric_col]:.4f}")
    print(f"  best:   {metric_col} {best[metric_col]:.4f} at step {int(best.step)}")
